In [7]:
import sys
sys.path.append('..')
import os
from src.BufferPool import Frame, BufferPool, FullCacheError
from src.FileManager import FileManager

print("Classe chamada.")

Classe chamada.


In [8]:
# SETUP: Criando o arquivo e populando o disco
caminho_arquivo = "teste_cache.bin"

# Apaga o arquivo se ele já existir (para o teste começar do zero)
if os.path.exists(caminho_arquivo):
    os.remove(caminho_arquivo)

print("Criando FileManager...")
fm = FileManager(caminho_arquivo)

print("Escrevendo páginas 1, 2 e 3 no disco...")
for i in range(1, 4):
    # Cria uma página vazia (4096 bytes) e coloca um "nome" nela
    page_vazia = bytearray(4095) 
    page_vazia[0:20] = f"Conteúdo da Pagina {i}".encode('utf-8')
    fm.write_in_page(i, page_vazia)

print("Disco populado com sucesso!\n")


Criando FileManager...
Escrevendo páginas 1, 2 e 3 no disco...
Disco populado com sucesso!



# Teste 1 - Hit

In [9]:
print("--- TESTE 1: HIT ---")
pool = BufferPool(fm, pool_size=3)
print("Primeira leitura (Deve ser MISS):")
pool.fetch_page(1)

print("Segunda leitura (Deve ser HIT):")
pool.fetch_page(1)
print("Se apareceu HIT acima, o teste passou!\n")

--- TESTE 1: HIT ---
Primeira leitura (Deve ser MISS):
MISS: Página não encontrada na Cache.
Segunda leitura (Deve ser HIT):
HIT: Página encontrada na Cache.
Se apareceu HIT acima, o teste passou!



# Teste 2 - Expulsão

In [10]:
print("--- TESTE 2: EXPULSÃO ---")
print("Lendo páginas 2 e 3 para encher o cache...")
pool.fetch_page(2)
pool.fetch_page(3)

print(f"LRU atual (deve ser [1, 2, 3]): {pool.lru_list}")

print("Lendo página 4 (o cache está cheio, deve expulsar a 1)...")
pool.fetch_page(4) 

print(f"LRU após expulsão (deve ser [2, 3, 4]): {pool.lru_list}")
print(f"Páginas no dicionário (deve ter 2, 3, 4): {list(pool.pages.keys())}")
print("Se a página 1 não está mais no dicionário, o teste passou!\n")


--- TESTE 2: EXPULSÃO ---
Lendo páginas 2 e 3 para encher o cache...
MISS: Página não encontrada na Cache.
MISS: Página não encontrada na Cache.
LRU atual (deve ser [1, 2, 3]): [1, 2, 3]
Lendo página 4 (o cache está cheio, deve expulsar a 1)...
MISS: Página não encontrada na Cache.
LRU após expulsão (deve ser [2, 3, 4]): [2, 3, 4]
Páginas no dicionário (deve ter 2, 3, 4): [2, 3, 4]
Se a página 1 não está mais no dicionário, o teste passou!



# Teste 3 - Página suja

In [11]:
print("--- TESTE 3: PÁGINA SUJA ---")
frame_p2 = pool.pages[2]
frame_p2.dirty = True
frame_p2 = pool.pages[2]
frame_p2.dirty = True
frame_p2.page.data[16] = ord('X')  # Altera o primeiro byte de dado

print("Lendo página 5 para forçar a expulsão da página 2 (suja)...")
pool.fetch_page(5)

fm_verificador = FileManager(caminho_arquivo)
dados_disco = fm_verificador.read_page(2)
print(f"Primeiros bytes da página 2 no disco: {dados_disco[16:24]}") 
print("Se aparecer 'ALTERADO', o cache gravou a página suja no disco. Teste passou!\n")

--- TESTE 3: PÁGINA SUJA ---
Lendo página 5 para forçar a expulsão da página 2 (suja)...
MISS: Página não encontrada na Cache.
Primeiros bytes da página 2 no disco: bytearray(b'Xna 2\x00\x00\x00')
Se aparecer 'ALTERADO', o cache gravou a página suja no disco. Teste passou!



# Teste 4 - Cache cheio de pinados

In [12]:
print("--- TESTE 4: FULL CACHE ERROR ---")
for pid in pool.pages:
    pool.pages[pid].pin_count = 1

try:
    print("Tentando ler página 6 com todas as páginas pinadas...")
    pool.fetch_page(6)
    print("ERRO: Não lançou a exceção!")
except FullCacheError:
    print("OK! A exceção FullCacheError foi lançada corretamente. Teste passou!")

--- TESTE 4: FULL CACHE ERROR ---
Tentando ler página 6 com todas as páginas pinadas...
MISS: Página não encontrada na Cache.
OK! A exceção FullCacheError foi lançada corretamente. Teste passou!
